# Geospatial Concepts

## Overview
Geospatial databases store, query, and analyse data that has a location component — monitoring sites, species observations, watersheds, protected areas. This notebook covers the foundational concepts needed to work with spatial data in PostGIS and GeoPandas.

**Core concepts:**

| Concept | Description |
|---|---|
| **Geometry** | Mathematical representation of a spatial feature (Point, Line, Polygon) |
| **CRS / SRID** | Coordinate Reference System — defines what the coordinates mean |
| **EPSG code** | Numeric identifier for a CRS (e.g., EPSG:4326 = WGS 84 lat/lon) |
| **WKT** | Well-Known Text — human-readable geometry format |
| **WKB** | Well-Known Binary — compact binary storage format |
| **GeoJSON** | JSON-based geometry format for web APIs |

**Critical rule: always project to local CRS before computing distance or area.**

```python
# WRONG: degrees ≠ metres
sites.geometry.distance(other)  # on EPSG:4326

# RIGHT: metres
sites.to_crs('EPSG:32620').geometry.distance(other)  # UTM Zone 20N
```

---

In [ ]:
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely import wkt, wkb
import geopandas as gpd

print("=== Core geometry types ===")
print()

# Point
site = Point(-66.06, 45.95)
print(f"Point:      {site}")
print(f"  WKT:      {site.wkt}")
print(f"  x={site.x}, y={site.y}")

# LineString
river = LineString([(-67.5, 46.2), (-66.8, 46.0), (-66.06, 45.95)])
print(f"\nLineString: {river.wkt[:60]}...")
print(f"  length:   {river.length:.4f} degrees")

# Polygon
watershed = Polygon([(-67.0, 46.5), (-65.5, 46.5), (-65.5, 45.5), (-67.0, 45.5), (-67.0, 46.5)])
print(f"\nPolygon:    {watershed.wkt[:60]}...")
print(f"  area:     {watershed.area:.4f} sq degrees")
print(f"  centroid: {watershed.centroid.wkt}")

# MultiPolygon
prot_area = MultiPolygon([
    Polygon([(-66.5,46.2),(-66.0,46.2),(-66.0,45.8),(-66.5,45.8),(-66.5,46.2)]),
    Polygon([(-68.0,47.0),(-67.5,47.0),(-67.5,46.5),(-68.0,46.5),(-68.0,47.0)]),
])
print(f"\nMultiPolygon: {len(prot_area.geoms)} polygons")
print(f"  total area: {prot_area.area:.4f} sq degrees")

print()
print("Geometry type hierarchy:")
types = [
    ("Point",             "Single (x, y) coordinate pair"),
    ("MultiPoint",        "Collection of points"),
    ("LineString",        "Ordered sequence of points forming a line"),
    ("MultiLineString",   "Collection of line strings"),
    ("Polygon",           "Closed ring + optional holes"),
    ("MultiPolygon",      "Collection of polygons (e.g., island chain)"),
    ("GeometryCollection","Heterogeneous mix of any geometry types"),
]
for t, desc in types:
    print(f"  {t:<22s}  {desc}")


---
## Coordinate Reference Systems

In [ ]:
import pyproj
from shapely.geometry import Point
from shapely.ops import transform as shapely_transform
import functools

print("=== Coordinate Reference Systems (CRS) ===")
print()

crs_table = [
    ("EPSG:4326",  "WGS 84",          "Geographic (degrees lat/lon)", "GPS, web maps, raw data — global"),
    ("EPSG:3857",  "Web Mercator",     "Projected (metres)",          "Google Maps, OSM tiles — web display"),
    ("EPSG:32620", "UTM Zone 20N",     "Projected (metres)",          "Atlantic Canada — distance/area calcs"),
    ("EPSG:32619", "UTM Zone 19N",     "Projected (metres)",          "Nova Scotia / New Brunswick"),
    ("EPSG:4269",  "NAD83",            "Geographic (degrees)",        "Canadian federal data (similar to WGS84)"),
    ("EPSG:3979",  "Canada Atlas LCC", "Projected (metres)",          "Canada-wide equal-area analysis"),
]
print(f"  {'EPSG':<12s}  {'Name':<20s}  {'Type':<22s}  Use case")
print("  " + "-"*80)
for epsg, name, typ, use in crs_table:
    print(f"  {epsg:<12s}  {name:<20s}  {typ:<22s}  {use}")

print()
print("=== CRS transformation with pyproj ===")
# Site in WGS84 degrees
site_wgs84 = Point(-66.06, 45.95)   # lon, lat — Moncton NB
print(f"WGS84 (degrees):    {site_wgs84.wkt}")

# Transform to UTM Zone 20N (metres)
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
x_utm, y_utm = transformer.transform(site_wgs84.x, site_wgs84.y)
site_utm = Point(x_utm, y_utm)
print(f"UTM Zone 20N (m):   POINT ({x_utm:.0f} {y_utm:.0f})")

# Buffer in metres (requires projected CRS)
buf_100m = site_utm.buffer(100)   # 100 m radius
buf_back = Point(*transformer.transform(buf_100m.centroid.x, buf_100m.centroid.y, direction='INVERSE'))
print(f"100m buffer centroid (back to WGS84): {buf_back.wkt}")

print()
print("Critical rule: ALWAYS check CRS before computing distance or area")
print("  - EPSG:4326 distances are in degrees (not metres!)")
print("  - 1 degree ≈ 111 km at equator but varies with latitude")
print("  - For accurate distance/area: project to a local CRS first (UTM)")
print()

print("PostGIS CRS syntax:")
pg_crs = [
    ("ST_GeomFromText('POINT(-66.06 45.95)', 4326)",  "Create geometry with SRID 4326"),
    ("ST_SetSRID(geom, 4326)",                         "Set SRID on existing geometry"),
    ("ST_Transform(geom, 32620)",                      "Reproject to EPSG:32620 (UTM 20N)"),
    ("ST_SRID(geom)",                                  "Get the SRID of a geometry"),
    ("ST_Distance(ST_Transform(a,32620), ST_Transform(b,32620))",  "Accurate metre distance"),
]
for sql, desc in pg_crs:
    print(f"  {sql}")
    print(f"    -> {desc}")


---
## WKT, WKB, and GeoJSON

In [ ]:
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping
import json

print("=== WKT, WKB, GeoJSON: geometry serialisation formats ===")
print()

from shapely.geometry import Point, Polygon

site = Point(-66.06, 45.95)
poly = Polygon([(-67.0,46.5),(-65.5,46.5),(-65.5,45.5),(-67.0,45.5)])

# WKT — Well-Known Text
print("WKT (Well-Known Text):")
print(f"  Point:   {site.wkt}")
print(f"  Polygon: {poly.wkt}")
print("  -- Human-readable; used in SQL: ST_GeomFromText('POINT(-66.06 45.95)', 4326)")

# WKB — Well-Known Binary
wkb_hex = site.wkb_hex
print(f"\nWKB hex (first 20 chars): {wkb_hex[:20]}...")
print("  -- Compact binary; used for storage; PostGIS stores geometry as WKB internally")

# GeoJSON
geojson = json.dumps(mapping(site), indent=2)
print(f"\nGeoJSON:")
print(geojson)
print("  -- JSON format; used in web APIs and frontends")

# Reading back
from shapely import wkt
recovered = wkt.loads("POINT (-66.06 45.95)")
print(f"\nRound-trip WKT -> Shapely -> WKT: {recovered.wkt}")

print()
print("Format comparison:")
formats = [
    ("WKT",     "POINT (-66.06 45.95)",  "Human-readable SQL input/output"),
    ("WKB",     "0101000000...",         "Binary storage (PostGIS internal)"),
    ("EWKT",    "SRID=4326;POINT(...)",  "WKT + SRID (PostGIS extension)"),
    ("GeoJSON", '{"type":"Point",...}',  "Web APIs, JavaScript, GeoDataFrame I/O"),
    ("Shapefile",".shp + .dbf + .prj",  "Legacy GIS format (ArcGIS, QGIS)"),
    ("GeoPackage",".gpkg",              "Modern SQLite-based spatial format"),
]
print(f"  {'Format':<12s}  {'Example':<30s}  Use case")
print("  " + "-"*70)
for fmt, ex, use in formats:
    print(f"  {fmt:<12s}  {ex:<30s}  {use}")


---
## Common Pitfalls

**1. Confusing longitude and latitude order**
PostGIS and Shapely use `(longitude, latitude)` order — i.e., `(x, y)` — matching mathematical convention. But many data sources (GPS, Google Maps, most geocoders) use `(latitude, longitude)`. `POINT(-66.06 45.95)` is Moncton NB (correct); `POINT(45.95 -66.06)` places the point in Kazakhstan. Always verify with a known landmark.

**2. Computing distances in degrees instead of metres**
`ST_Distance(a, b)` on EPSG:4326 geometries returns degrees, not metres. 1 degree ≈ 111 km at the equator, but varies with latitude. Always project to a local CRS (UTM) before distance/area calculations: `ST_Distance(ST_Transform(a, 32620), ST_Transform(b, 32620))`.

**3. Using EPSG:3857 (Web Mercator) for area/distance calculations**
Web Mercator heavily distorts areas at high latitudes. A polygon in New Brunswick appears much larger in EPSG:3857 than it actually is. Use UTM or an equal-area projection (EPSG:3979 for Canada) for any area or distance work.

**4. Storing coordinates as TEXT instead of geometry type**
Storing `lat = '45.95'` and `lon = '-66.06'` as TEXT columns means no spatial indexes, no PostGIS functions, and manual haversine formulas. Use `geometry(Point, 4326)` or `geography` column types from the start.

**5. Not setting SRID when creating geometries**
`ST_GeomFromText('POINT(-66.06 45.95)')` creates a geometry with SRID=0 (unknown). `ST_Transform` fails on SRID=0. Always specify the SRID: `ST_GeomFromText('POINT(-66.06 45.95)', 4326)` or `ST_SetSRID(geom, 4326)`.

**6. Mixing geometry and geography types**
PostGIS has both `geometry` (planar math, CRS-aware) and `geography` (sphere math, always degrees). `ST_Distance(geometry, geography)` raises a type mismatch error. Cast explicitly: `ST_Distance(a::geography, b::geography)` for accurate global distances without reprojection.


---
*sql_methods_library - Samantha McGarrigle*